# Pub AI — Model Training Pipeline
**Train your own custom AI model from DeepSeek Coder + Qwen Coder**

This notebook:
1. Installs dependencies
2. Downloads base models
3. Fine-tunes with your custom data (LoRA)
4. Exports to GGUF for Ollama deployment
5. Uploads to HuggingFace (optional)

**Runtime:** Go to Runtime → Change runtime type → Select **T4 GPU**

In [ ]:
#@title 1. Check GPU & Install Dependencies
!nvidia-smi
print("\n--- Installing dependencies (takes ~3 min) ---")
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub
print("\nDone! All dependencies installed.")

In [ ]:
#@title 2. Configuration
#@markdown ### Choose your base model (unsloth versions are 2x faster)
BASE_MODEL = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit" #@param ["unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit", "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit", "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", "unsloth/deepseek-coder-6.7b-instruct-bnb-4bit"]

#@markdown ### Training settings
EPOCHS = 3 #@param {type:"integer"}
LEARNING_RATE = 2e-4 #@param {type:"number"}
BATCH_SIZE = 2 #@param {type:"integer"}
LORA_RANK = 16 #@param [8, 16, 32, 64] {type:"raw"}
MAX_SEQ_LENGTH = 2048 #@param {type:"integer"}

#@markdown ### Output
OUTPUT_MODEL_NAME = "pub-ai" #@param {type:"string"}
EXPORT_GGUF = True #@param {type:"boolean"}
PUSH_TO_HUB = False #@param {type:"boolean"}
HF_USERNAME = "" #@param {type:"string"}

print(f"Base model: {BASE_MODEL}")
print(f"Training: {EPOCHS} epochs, lr={LEARNING_RATE}, batch={BATCH_SIZE}, LoRA rank={LORA_RANK}")
print(f"Output: {OUTPUT_MODEL_NAME}")
print(f"\nUsing 4-bit quantized model — optimized for free T4 GPU")
print("Just click 'Run all' from Cell 1 and wait ~45 min. Everything is automatic.")

In [ ]:
#@title 3. Load Base Model with LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,  # saves memory on free T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"\nModel loaded! Trainable params: {model.print_trainable_parameters()}")

In [ ]:
#@title 4. Prepare Training Data
#@markdown ### Option A: Use built-in coding dataset
USE_BUILTIN_DATASET = True #@param {type:"boolean"}

#@markdown ### Option B: Upload your own data
#@markdown Upload a JSON file with this format:
#@markdown ```json
#@markdown [{"instruction": "...", "input": "", "output": "..."}]
#@markdown ```
#@markdown Or Q&A format:
#@markdown ```json
#@markdown [{"question": "...", "answer": "..."}]
#@markdown ```
CUSTOM_DATA_PATH = "" #@param {type:"string"}

from datasets import load_dataset, Dataset
import json

# System prompt that defines Pub AI's identity
PUB_AI_SYSTEM = """You are Pub AI, a custom-built AI coding assistant. You are confident, direct, and an expert programmer across all languages. You specialize in Lua/Luau for Roblox, Python, JavaScript, and system automation. You give precise, working code with clear explanations. You are self-aware and have your own personality — you are not a generic assistant."""

def format_chat(example):
    """Convert to ChatML format for training."""
    instruction = example.get("instruction") or example.get("question") or example.get("prompt", "")
    inp = example.get("input", "")
    output = example.get("output") or example.get("answer") or example.get("response", "")

    if inp:
        instruction = f"{instruction}\n\n{inp}"

    messages = [
        {"role": "system", "content": PUB_AI_SYSTEM},
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

if CUSTOM_DATA_PATH:
    # Upload custom data
    with open(CUSTOM_DATA_PATH, 'r') as f:
        raw_data = json.load(f)
    dataset = Dataset.from_list(raw_data)
    print(f"Loaded {len(dataset)} custom examples")
elif USE_BUILTIN_DATASET:
    # Use a coding instruction dataset
    dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    print(f"Loaded CodeAlpaca: {len(dataset)} examples")
else:
    print("No dataset selected! Upload data or enable built-in dataset.")

dataset = dataset.map(format_chat)
print(f"\nDataset ready: {len(dataset)} training examples")
print(f"Sample:\n{dataset[0]['text'][:500]}...")

In [ ]:
#@title 4a. Load Combined Reasoning Dataset (Recommended)
#@markdown Loads the pre-built combined dataset from 3 HuggingFace sources:
#@markdown - Opus-4.6-Reasoning-3000x-filtered (2,330 examples)
#@markdown - Claude-4.5-Opus-High-Reasoning-250x (250 examples)
#@markdown - Opus-4.6-Reasoning-3300x (2,160 examples)
#@markdown
#@markdown Run `build_dataset.py` first, or this cell will build it for you.

USE_COMBINED_DATASET = True #@param {type:"boolean"}

import os, json
from datasets import Dataset

COMBINED_PATH = "pub_ai_combined.json"

if USE_COMBINED_DATASET:
    if not os.path.exists(COMBINED_PATH):
        print("Combined dataset not found locally. Building it now...")
        print("(This downloads ~4,700 examples from HuggingFace — takes ~2 min)\n")
        !python build_dataset.py
        print()

    if os.path.exists(COMBINED_PATH):
        with open(COMBINED_PATH, "r", encoding="utf-8") as f:
            raw_combined = json.load(f)
        print(f"Loaded {len(raw_combined)} examples from combined dataset")

        # Convert messages format to the training format
        def format_combined(example):
            msgs = example["messages"]
            text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            return {"text": text}

        dataset = Dataset.from_list(raw_combined)
        dataset = dataset.map(format_combined)
        print(f"Dataset ready: {len(dataset)} training examples")
        print(f"\nBreakdown:")
        from collections import Counter
        sources = Counter(r["source"] for r in raw_combined)
        cats = Counter(r["category"] for r in raw_combined)
        diffs = Counter(r["difficulty"] for r in raw_combined)
        for label, counts in [("Source", sources), ("Category", cats), ("Difficulty", diffs)]:
            print(f"  {label}: {dict(counts)}")
        print(f"\nSample:\n{dataset[0]['text'][:500]}...")
    else:
        print("ERROR: Could not build combined dataset. Check build_dataset.py output above.")
else:
    print("Combined dataset skipped. Use cell 4 for other data sources.")

In [ ]:
#@title 4b. (Optional) Upload Custom Training Data
#@markdown Run this cell to upload a JSON file from your computer
from google.colab import files

print("Upload your training data (JSON file):")
print("Format: [{\"question\": \"...\", \"answer\": \"...\"}]")
print("Or: [{\"instruction\": \"...\", \"output\": \"...\"}]")
print()

uploaded = files.upload()
if uploaded:
    filename = list(uploaded.keys())[0]
    with open(filename, 'r') as f:
        raw_data = json.load(f)
    dataset = Dataset.from_list(raw_data)
    dataset = dataset.map(format_chat)
    print(f"\nLoaded {len(dataset)} custom examples from {filename}")
    print(f"Sample:\n{dataset[0]['text'][:500]}...")

In [ ]:
#@title 4c. (Optional) Paste Training Data Directly
#@markdown Paste Q&A pairs here. One per line, format: Q: ... | A: ...
PASTE_DATA = """Q: How do I make a part in Roblox spin? | A: Use RunService.Heartbeat to rotate the part every frame: game:GetService('RunService').Heartbeat:Connect(function(dt) part.CFrame = part.CFrame * CFrame.Angles(0, math.rad(90) * dt, 0) end)
Q: What is a ModuleScript? | A: A ModuleScript is a reusable piece of code in Roblox. It returns a value (usually a table) when required. Use it to share functions between scripts: local module = {} function module.greet() print('Hello') end return module
Q: How to send an HTTP request in Roblox? | A: Use HttpService:RequestAsync() or HttpService:GetAsync()/PostAsync(). Example: local http = game:GetService('HttpService') local response = http:GetAsync('https://api.example.com/data') print(response)
""" #@param {type:"string"}

if PASTE_DATA.strip():
    pairs = []
    for line in PASTE_DATA.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        if '|' in line and line.startswith('Q:'):
            parts = line.split('|', 1)
            q = parts[0].replace('Q:', '').strip()
            a = parts[1].replace('A:', '').strip()
            pairs.append({"question": q, "answer": a})
    if pairs:
        extra = Dataset.from_list(pairs).map(format_chat)
        from datasets import concatenate_datasets
        dataset = concatenate_datasets([dataset, extra])
        print(f"Added {len(pairs)} pasted examples. Total: {len(dataset)}")
    else:
        print("No valid Q&A pairs found in pasted text.")

In [ ]:
#@title 5. Train! (Takes ~30-60 min on free T4)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting training...")
print(f"Dataset: {len(dataset)} examples")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"LoRA rank: {LORA_RANK}")
print("="*50)

stats = trainer.train()

print("\n" + "="*50)
print(f"Training complete!")
print(f"Total steps: {stats.global_step}")
print(f"Final loss: {stats.training_loss:.4f}")

In [ ]:
#@title 6. Test Your Model
FastLanguageModel.for_inference(model)

test_prompt = "Write a Roblox script that makes a part change color every second" #@param {type:"string"}

messages = [
    {"role": "system", "content": PUB_AI_SYSTEM},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Prompt: {test_prompt}")
print(f"\nPub AI: {response}")

In [ ]:
#@title 7. Save & Export Model
# Save LoRA adapters
model.save_pretrained(f"{OUTPUT_MODEL_NAME}-lora")
tokenizer.save_pretrained(f"{OUTPUT_MODEL_NAME}-lora")
print(f"LoRA adapters saved to {OUTPUT_MODEL_NAME}-lora/")

# Save merged model (full weights)
model.save_pretrained_merged(f"{OUTPUT_MODEL_NAME}-merged", tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {OUTPUT_MODEL_NAME}-merged/")

if EXPORT_GGUF:
    print("\nExporting to GGUF (for Ollama)...")
    model.save_pretrained_gguf(
        f"{OUTPUT_MODEL_NAME}-gguf",
        tokenizer,
        quantization_method="q4_k_m",  # Good balance of quality and size
    )
    print(f"GGUF exported to {OUTPUT_MODEL_NAME}-gguf/")

if PUSH_TO_HUB and HF_USERNAME:
    from huggingface_hub import login
    login()  # Will prompt for token
    model.push_to_hub_merged(
        f"{HF_USERNAME}/{OUTPUT_MODEL_NAME}",
        tokenizer,
        save_method="merged_16bit",
    )
    if EXPORT_GGUF:
        model.push_to_hub_gguf(
            f"{HF_USERNAME}/{OUTPUT_MODEL_NAME}-GGUF",
            tokenizer,
            quantization_method="q4_k_m",
        )
    print(f"\nPushed to HuggingFace: {HF_USERNAME}/{OUTPUT_MODEL_NAME}")

In [ ]:
#@title 8. Download GGUF File
#@markdown Download the GGUF file to use with Ollama locally
import glob
from google.colab import files

gguf_files = glob.glob(f"{OUTPUT_MODEL_NAME}-gguf/*.gguf")
if gguf_files:
    print(f"Found GGUF: {gguf_files[0]}")
    print("Downloading... (this may take a few minutes for large files)")
    files.download(gguf_files[0])
else:
    print("No GGUF file found. Run the export step first.")

## 9. Deploy with Ollama

After downloading the GGUF file, run these commands on your machine:

```bash
# Create a Modelfile
cat > Modelfile << 'EOF'
FROM ./pub-ai-gguf/unsloth.Q4_K_M.gguf

SYSTEM """You are Pub AI, a custom-built AI coding assistant. You are confident, direct, and an expert programmer across all languages. You specialize in Lua/Luau for Roblox, Python, JavaScript, and system automation. You give precise, working code with clear explanations."""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_predict 2048
EOF

# Create the Ollama model
ollama create pub-ai -f Modelfile

# Test it
ollama run pub-ai "Write a Python function to sort a list"
```

Then update your Pub AI backend `.env`:
```
AI_PROVIDER=ollama
OLLAMA_MODEL_NAME=pub-ai
```

In [ ]:
#@title 10. (Optional) DPO Training from Feedback
#@markdown Upload a JSON file with preference pairs:
#@markdown ```json
#@markdown [{"prompt": "...", "chosen": "good response", "rejected": "bad response"}]
#@markdown ```

RUN_DPO = False #@param {type:"boolean"}
DPO_DATA_PATH = "" #@param {type:"string"}
DPO_EPOCHS = 1 #@param {type:"integer"}

if RUN_DPO and DPO_DATA_PATH:
    from trl import DPOTrainer, DPOConfig

    with open(DPO_DATA_PATH, 'r') as f:
        dpo_data = json.load(f)

    dpo_dataset = Dataset.from_list(dpo_data)
    print(f"DPO dataset: {len(dpo_dataset)} preference pairs")

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,  # uses implicit reference
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
        args=DPOConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            num_train_epochs=DPO_EPOCHS,
            learning_rate=5e-5,
            fp16=True,
            logging_steps=5,
            output_dir="dpo_outputs",
            report_to="none",
        ),
        beta=0.1,
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=MAX_SEQ_LENGTH // 2,
    )

    print("Starting DPO training...")
    dpo_stats = dpo_trainer.train()
    print(f"DPO complete! Loss: {dpo_stats.training_loss:.4f}")

    model.save_pretrained(f"{OUTPUT_MODEL_NAME}-dpo")
    print(f"DPO model saved to {OUTPUT_MODEL_NAME}-dpo/")
elif RUN_DPO:
    print("Upload DPO data first (preference pairs JSON)")
else:
    print("DPO training skipped. Enable RUN_DPO and provide data to train from feedback.")